<a href="https://colab.research.google.com/github/pratripat/English-to-Hindi-Translator/blob/main/EngToHindi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Load raw corpus

In [ ]:
from datasets import load_dataset

In [ ]:
# IITB English-Hindi parallel corpus, standard benchmark for this task.
# ~1.66M train pairs, ~2500 val, ~2500 test.
raw = load_dataset("cfilt/iitb-english-hindi")

In [ ]:
print(raw['train'][0])

## Step 2: Clean and filter the corpus to something that can be used

In [ ]:
import re
from tqdm import tqdm

In [ ]:
# Cleans a given text
def basic_clean(text: str) -> str:
  text = text.strip() # Remove trailing white spaces
  text = re.sub(r"\s+", " ", text) # This collapses multiple spaces, tabs into one space. This helps the model retain context or make multiple tokens for the spaces and tabs.
  return text

In [ ]:
# Filters out the pairs that are actually relevant
def filter_pair(en: str, hi: str, min_len=1, max_len=40, max_ratio=2.5):
  en_toks = en.split()
  hi_toks = hi.split()

  # There should be atleast one token that will be converted into hindi tokens.
  # Max length is kept because the conversion is a costly task O(n^2)
  if not en_toks or not hi_toks:
      return False
  if len(en_toks) < min_len or len(hi_toks) < min_len:
      return False
  if len(en_toks) > max_len or len(hi_toks) > max_len:
      return False

  # When some english sentence is converted into another language, it is usually relatively similar size
  # in terms of tokens, if we have 50 words being converted to 3 words, then the conversion is probably wrong
  # the data fetching might have problems or the data set might have some irrelevant conversions
  # (sentence alignment error)
  ratio = len(en_toks) / len(hi_toks)
  if ratio > max_ratio or ratio < (1 / max_ratio):
      return False

  return True

In [ ]:
def clean_and_filter_split(split, max_len=40, max_ratio=2.5):
  en_lines = []
  hi_lines = []
  seen = set()

  for ex in tqdm(split, desc='cleaning'):
    en = basic_clean(ex['translation']['en'])
    hi = basic_clean(ex['translation']['hi'])

    if not filter_pair(en, hi, max_len=max_len, max_ratio=max_ratio):
      continue

    if (en, hi) in seen:
      continue

    seen.add((en, hi))
    en_lines.append(en)
    hi_lines.append(hi)

  return en_lines, hi_lines

In [ ]:
DEV_SUBSET_SIZE = 100000 # Starting small just for pipelining sake

In [ ]:
train_subset = raw["train"].select(range(min(DEV_SUBSET_SIZE, len(raw["train"]))))

train_en, train_hi = clean_and_filter_split(train_subset)
val_en, val_hi = clean_and_filter_split(raw['validation'])
test_en, test_hi = clean_and_filter_split(raw['test'])

In [ ]:
print(f"train pairs kept: {len(train_en)}")
print(f"val pairs kept:   {len(val_en)}")
print(f"test pairs kept:  {len(test_en)}")

# Step3: Writing the data into plain files for SentencePiece training

In [ ]:
import os

In [ ]:
os.makedirs("data", exist_ok=True)

def write_lines(lines, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

write_lines(train_en, "data/train.en")
write_lines(train_hi, "data/train.hi")
write_lines(val_en, "data/val.en")
write_lines(val_hi, "data/val.hi")
write_lines(test_en, "data/test.en")
write_lines(test_hi, "data/test.hi")

# Step 4: Training the tokenizers

In [ ]:
import sentencepiece as spm

We will be using two different vocabs for english and hindi. This is because we will never encounter a scenario where we need both the english and hindi contents together. So there is no point forcing both the tokens to be in one table. The encoder uses the english vocab and the decoder uses the hindi vocab (for generation) along with the english vocab.

In [ ]:
VOCAB_SIZE_EN = 8000
VOCAB_SIZE_HI = 8000

In [ ]:
def train_sentencepiece(input_file, model_prefix, vocab_size):
    spm.SentencePieceTrainer.train(
        input=input_file,
        model_prefix=model_prefix,
        vocab_size=vocab_size,
        model_type="bpe", # byte pair encoding
        character_coverage=0.9995,  # important for Hindi: keeps rare Devanagari chars
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3,
        pad_piece="<pad>",
        unk_piece="<unk>",
        bos_piece="<bos>",
        eos_piece="<eos>",
    )

In [ ]:
train_sentencepiece("data/train.en", "data/spm_en", VOCAB_SIZE_EN)
train_sentencepiece("data/train.hi", "data/spm_hi", VOCAB_SIZE_HI)

In [ ]:
sp_en = spm.SentencePieceProcessor(model_file="data/spm_en.model")
sp_hi = spm.SentencePieceProcessor(model_file="data/spm_hi.model")

In [ ]:
# Sanity check -- always eyeball this before trusting the tokenizer
sample_en = train_en[0]
sample_hi = train_hi[0]
print("EN:", sample_en)
print("EN tokens:", sp_en.encode(sample_en, out_type=str))
print("HI:", sample_hi)
print("HI tokens:", sp_hi.encode(sample_hi, out_type=str))

The tokens are being created properly (as it should have)